In [ ]:
# Data Processing (Convert JSON to CSV)

import os
import json
import pandas as pd


data_folder = "DSE"   


securities_path = os.path.join(data_folder, "securities.json")
securities_df = pd.read_json(securities_path)

# remove leading/trailing spaces in trading_code
securities_df["trading_code"] = securities_df["trading_code"].astype(str).str.strip().str.upper()

# Keep only needed cols (optional)
securities_df = securities_df[["trading_code", "sector", "instrument_type"]]


all_rows = []

for file in os.listdir(data_folder):
    if file.startswith("prices_") and file.endswith(".json"):
        year = int(file.split("_")[1].split(".")[0])
        file_path = os.path.join(data_folder, file)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for row in data:
            # add year
            row["year"] = year

            # clean date: remove "00:00:00"
            if "date" in row and isinstance(row["date"], str):
                row["date"] = row["date"].split(" ")[0]

            # clean trading_code (strip spaces)
            if "trading_code" in row:
                row["trading_code"] = str(row["trading_code"]).strip().upper()

        all_rows.extend(data)

prices_df = pd.DataFrame(all_rows)


df = prices_df.merge(securities_df, on="trading_code", how="left")

df["sector"] = df["sector"].fillna("Unknown")
df["instrument_type"] = df["instrument_type"].fillna("Unknown")

output_path = os.path.join(data_folder, "dse.csv")
df.to_csv(output_path, index=False)

print("✅ Done!")
print("📁 Saved:", output_path)
print("📊 Rows:", len(df))
print("🔎 Missing sector:", (df["sector"] == "Unknown").sum())

✅ Done!
📁 Saved: DSE\dse.csv
📊 Rows: 1791069
🔎 Missing sector: 129412


In [ ]:
# Feature Engineering

import pandas as pd
import numpy as np

file_path = 'DSE\\dse.csv' 
df = pd.read_csv(file_path)

# Convert 'date' column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract day, month, and year from the date
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

# Calculate daily return
df['daily_return'] = (df['closing_price'] - df['yesterdays_closing_price']) / df['yesterdays_closing_price'] * 100

# Calculate High-Low spread
df['high_low_spread'] = df['high'] - df['low']

# Calculate Intraday Volatility
df['intraday_volatility'] = (df['high'] - df['low']) / df['closing_price'] * 100

# Calculate Volume Change %
df['volume_change_pct'] = (df['volume'] - df['volume'].shift(1)) / df['volume'].shift(1) * 100

# Calculate Price Gap (Opening - Yesterday's Closing)
df['price_gap'] = df['opening_price'] - df['yesterdays_closing_price']

# Calculate rolling 5-day volatility
df['rolling_5_day_volatility'] = df['daily_return'].rolling(window=5).std()

# Calculate rolling 10-day momentum
df['rolling_10_day_momentum'] = df['closing_price'] - df['closing_price'].shift(10)

# Create target variable (market type)
conditions = [
    df['daily_return'] <= -10,  # Crash day (price drop over 10%)
    df['high_low_spread'] > 5,  # High volatility day (spread > 5%)
    df['daily_return'] > 0      # Normal day
]
choices = [2, 1, 0]
df['market_type'] = np.select(conditions, choices, default=0)

# Drop rows with NaN values generated from shift operations and rolling calculations
df.dropna(inplace=True)

# Export the processed dataframe to a new CSV file
output_path = 'dse_market_stress.csv'  # Update the desired output file path
df.to_csv(output_path, index=False)

print(f"Processed dataset has been saved to: {output_path}")

Processed dataset has been saved to: dse_market_stress.csv
